# Simple Study Set Builder

This notebook is a small test harness for the Python Library Adapter. It creates simple Study Sets made only from explicit session references.

Workflow:

1. Choose the target processed library using the runtime settings panel.
2. Build the existing BODAQS session selector for that library.
3. Select one or more sessions.
4. Create a Study Set from those sessions.

If the selector includes aggregations, this notebook intentionally flattens the current selection to physical sessions before saving the Study Set.

In [1]:
from pathlib import Path
import sys

import ipywidgets as W
import pandas as pd
from IPython.display import display

# Make the local analysis package importable when this notebook is run from
# analysis/Library Adapter testing.
ANALYSIS_ROOT = Path.cwd()
if not (ANALYSIS_ROOT / "bodaqs_analysis").exists():
    ANALYSIS_ROOT = Path.cwd().parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from bodaqs_analysis.ui import make_preprocess_runtime_settings_editor
from bodaqs_analysis.widgets.session_selector import make_session_selector
from bodaqs_analysis.library_api import LibraryAdapter

runtime_settings_editor = make_preprocess_runtime_settings_editor(
    show_log_dir=False,
    show_preprocess_profile_path=False,
    show_generic_log_metadata=False,
    show_logger_timezone=False,
    show_bike_profile_path=False,
    show_fit_inputs=False,
    show_prompt_for_descriptions=False,
    show_run_tz_label=False,
)

display(runtime_settings_editor.ui)

## Build The Session Selector

Run this cell after choosing the target artifacts/library directory above. The adapter expects a common libraries root, so the helper below resolves that from the selected artifacts directory.

In [3]:
def resolve_library_adapter_from_artifacts_dir(artifacts_dir: str | Path):
    artifacts_path = Path(artifacts_dir).expanduser().resolve()
    candidate_roots = [artifacts_path.parent, artifacts_path]
    errors = []

    for libraries_root in candidate_roots:
        try:
            adapter = LibraryAdapter(libraries_root)
            libraries = adapter.list_libraries(refresh=True)
        except Exception as exc:
            errors.append(f"{libraries_root}: {type(exc).__name__}: {exc}")
            continue

        for library in libraries:
            if Path(library["root"]).expanduser().resolve() == artifacts_path:
                return adapter, library

    raise RuntimeError(
        "Could not resolve the selected artifacts directory as a LibraryAdapter library.\n"
        f"Selected artifacts_dir: {artifacts_path}\n"
        "Tried libraries roots:\n- " + "\n- ".join(errors or map(str, candidate_roots))
    )


runtime_settings = runtime_settings_editor.get_settings()
ARTIFACTS_DIR = Path(runtime_settings["artifacts_dir"]).expanduser().resolve()
adapter, library = resolve_library_adapter_from_artifacts_dir(ARTIFACTS_DIR)
LIBRARY_ID = library["library_id"]

print(f"Artifacts directory: {ARTIFACTS_DIR}")
print(f"Resolved library: {library['display_name']} ({LIBRARY_ID})")
print(f"Libraries root: {adapter.libraries_root.resolve()}")

selector = make_session_selector(
    artifacts_dir=ARTIFACTS_DIR,
    select_first_by_default=False,
)
display(selector["ui"])

Artifacts directory: C:\Users\benco\OneDrive\BODAQS-data\default-library
Resolved library: Default Library (default-library)
Libraries root: C:\Users\benco\OneDrive\BODAQS-data


## Create A Session-Only Study Set

Select sessions in the selector above, enter a display name, then click **Create Study Set**. Leave the ID blank unless you specifically want to choose it; otherwise the adapter derives a safe unique ID from the display name.

In [4]:
study_set_name = W.Text(
    value="",
    description="Name",
    placeholder="e.g. Setup comparison 1",
    layout=W.Layout(width="640px"),
)
study_set_id = W.Text(
    value="",
    description="ID optional",
    placeholder="Leave blank to auto-generate",
    layout=W.Layout(width="640px"),
)
create_button = W.Button(description="Create Study Set", button_style="primary")
refresh_button = W.Button(description="Refresh list")
out = W.Output()

created_study_set = None
study_set_bridge = None
study_set_selector_handle = None


def selected_session_refs():
    key_to_ref = selector["get_key_to_ref"]()
    refs = []
    for session_key, (run_id, session_id) in key_to_ref.items():
        refs.append(
            {
                "session_key": str(session_key),
                "run_id": str(run_id),
                "session_id": str(session_id),
                "label": str(session_key),
            }
        )
    return refs


def study_sets_df():
    rows = adapter.list_study_sets(LIBRARY_ID)
    return pd.DataFrame(rows) if rows else pd.DataFrame(
        columns=["study_set_id", "display_name", "revision", "updated_at", "session_count", "path"]
    )


def show_study_sets():
    display(study_sets_df())


def on_create_clicked(_):
    global created_study_set, study_set_bridge, study_set_selector_handle
    with out:
        out.clear_output()
        display_name = study_set_name.value.strip()
        if not display_name:
            print("Enter a Study Set name first.")
            return

        sessions = selected_session_refs()
        if not sessions:
            print("Select at least one session first.")
            return

        payload = {
            "display_name": display_name,
            "sessions": sessions,
        }
        if study_set_id.value.strip():
            payload["study_set_id"] = study_set_id.value.strip()

        try:
            created_study_set = adapter.create_study_set(LIBRARY_ID, payload)
            study_set_bridge = adapter.study_set_to_selection_snapshot(
                LIBRARY_ID,
                created_study_set["study_set_id"],
                include_groupings=False,
            )
            study_set_selector_handle = study_set_bridge["selector_handle"]
        except Exception as exc:
            print(f"Create failed: {type(exc).__name__}: {exc}")
            return

        print(
            "Created Study Set "
            f"{created_study_set['display_name']!r} "
            f"({created_study_set['study_set_id']}) "
            f"with {len(created_study_set['sessions'])} session(s)."
        )
        print("The variable study_set_selector_handle is now available for widget testing.")
        show_study_sets()


def on_refresh_clicked(_):
    with out:
        out.clear_output()
        show_study_sets()


create_button.on_click(on_create_clicked)
refresh_button.on_click(on_refresh_clicked)

display(W.VBox([study_set_name, study_set_id, W.HBox([create_button, refresh_button]), out]))
with out:
    show_study_sets()

## Optional: Load An Existing Study Set As A Selector Handle

Use this if you want to test an existing Study Set without creating a new one. Set `TARGET_STUDY_SET_ID`, run the cell, then pass `study_set_selector_handle` into widgets that accept a session selector handle.

In [5]:
TARGET_STUDY_SET_ID = "test-study-set-1"  # e.g. "setup-comparison-1"

if TARGET_STUDY_SET_ID.strip():
    study_set_bridge = adapter.study_set_to_selection_snapshot(
        LIBRARY_ID,
        TARGET_STUDY_SET_ID.strip(),
        include_groupings=False,
    )
    study_set_selector_handle = study_set_bridge["selector_handle"]
    display(study_set_bridge["events_index_df"])
    print(f"Loaded Study Set: {study_set_bridge['display_name']} ({study_set_bridge['study_set_id']})")
else:
    print("Set TARGET_STUDY_SET_ID first if you want to load an existing Study Set.")

sel=study_set_selector_handle

,session_key,run_id,session_id
0,run_2026-05-28T10-55-12_LOCAL::2026-02-19_09-4...,run_2026-05-28T10-55-12_LOCAL,2026-02-19_09-43-31
1,run_2026-05-28T10-54-56_LOCAL::2026-02-19_09-3...,run_2026-05-28T10-54-56_LOCAL,2026-02-19_09-37-10
2,run_2026-05-28T10-54-40_LOCAL::2026-02-19_08-5...,run_2026-05-28T10-54-40_LOCAL,2026-02-19_08-59-25
3,run_2026-05-28T10-54-28_LOCAL::2026-02-19_08-5...,run_2026-05-28T10-54-28_LOCAL,2026-02-19_08-57-46


Loaded Study Set: Test study set 1 (test-study-set-1)


In [8]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_widget_for_loader
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_rebuilder

key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
 # session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)

hist = make_signal_histogram_rebuilder(sel=sel)
display(hist["out"])


Output()